# 04. ResNet-18 + CLIP 하이브리드 추천 (최종 학습 버전)

- ImageNet 사전학습 ResNet-18 백본 + 스타일 분류 헤드(6) / 향조 헤드(5) 멀티태스크 구조, 스타일 헤드만 클래스 가중 CE로 학습 (Adam 5e-5, 4 epoch, 증강 없음)
- 추천: ResNet 스타일 확률 × 매핑 행렬 → 후보 향수 10개의 향조 점수 (0.45) + CLIP 이미지-향수명 유사도 (0.55)
- 결과: [`results/pred_hybrid_clip_resnet.csv`](../results/pred_hybrid_clip_resnet.csv)

> **참고 (README 회고 참조)**
> - 다른 모델과 달리 스타일 라벨을 `usage` 컬럼만으로 다시 매핑해 학습했습니다 (`smart casual → Street`, `party → Romantic`).
> - `mapping_df`(행: Romantic, Casual, …)와 `STYLE_CLASSES`(Casual, Formal, …), 매핑 열 순서와 `ACCORD_CLASSES`의 순서가 서로 달라 행렬곱 결과가 엇갈려 매칭됩니다.
> - 추천 후보는 24,063개 전체가 아니라 코드에 직접 지정한 10개 향수입니다. `hybrid_top1_score`는 0.73~0.96 범위로 재조정한 표시용 값입니다.
> 실행 당시 코드를 그대로 보존하기 위해 수정하지 않았습니다.


In [ ]:
%pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from google.colab import drive
import clip

# 1. 환경 설정 및 드라이브 마운트
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/FashionPerfumeProject'
IMAGE_DIR = os.path.join(PROJECT_ROOT, 'images')

# 파일 로드
train_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'train (1).csv'))
test_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'test (1).csv'))
mapping_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'style_accord_mapping.csv'), index_col=0)

# 2. 데이터 매핑 및 클래스 설정
STYLE_CLASSES = ['Casual', 'Formal', 'Romantic', 'Sporty', 'Street', 'Vintage']
USAGE_TO_IDX = {c: i for i, c in enumerate(STYLE_CLASSES)}
custom_mapping = {
    'casual': 'Casual', 'formal': 'Formal', 'sports': 'Sporty',
    'ethnic': 'Vintage', 'smart casual': 'Street', 'party': 'Romantic'
}
train_df['mapped_style'] = train_df['usage'].map(custom_mapping).fillna('Casual')
test_df['mapped_style'] = test_df['usage'].map(custom_mapping).fillna('Casual')

# 3. 모델 정의
class MultiTaskResNet18(nn.Module):
    def __init__(self, num_usages=6, num_accords=5):
        super(MultiTaskResNet18, self).__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.fc_usage = nn.Linear(num_ftrs, num_usages)
        self.fc_perfume = nn.Linear(num_ftrs, num_accords)

    def forward(self, x):
        features = self.backbone(x)
        usage_logits = self.fc_usage(features)
        perfume_scores = self.fc_perfume(features)
        return usage_logits, perfume_scores

# 4. 학습 파트
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiTaskResNet18().to(device)

# 가중치 계산
train_labels = train_df['mapped_style'].map(USAGE_TO_IDX).fillna(0).astype(int).values
class_counts = np.bincount(train_labels, minlength=6)
class_weights = 1.0 / (class_counts + 1e-5)
class_weights = class_weights / class_weights.sum()
weight_tensor = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = optim.Adam(model.parameters(), lr=5e-5)

# 학습 데이터 로더
class FashionDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{int(row['id'])}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.fromarray(np.uint8(np.random.rand(224, 224, 3) * 255))
        if self.transform: image = self.transform(image)
        label = USAGE_TO_IDX.get(row['mapped_style'], 0)
        return image, torch.tensor(label, dtype=torch.long), torch.tensor(row['id'], dtype=torch.long)

train_loader = DataLoader(FashionDataset(train_df, IMAGE_DIR, transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])), batch_size=32, shuffle=True)

print("🚀 학습 시작!")
model.train()
for epoch in range(4):
    for imgs, labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

torch.save(model.state_dict(), os.path.join(PROJECT_ROOT, 'final_model.pth'))
print("✅ 학습 완료 및 모델 저장.")

Mounted at /content/drive
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 191MB/s]


🚀 학습 시작!


Epoch 4: 100%|██████████| 669/669 [01:17<00:00,  8.61it/s]


✅ 학습 완료 및 모델 저장.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import clip

# 1. 테스트 데이터 로더 설정
test_loader = DataLoader(
    FashionDataset(
        test_df,
        IMAGE_DIR,
        transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    ),
    batch_size=32,
    shuffle=False
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model, preprocess = clip.load("ViT-B/32", device=device)

# 2. 오리지널 향수 명세 및 확장 맵 선언
fra_df = pd.DataFrame({
    'perfume': ['secret-obsession', 'declaration-d-un-soir-intense', 'soir-de-lune-edition-limitee-2015', 'montauk', 'new-york-gabriela-hearst', 'the-poplin-shirt', 'aatifa', 'annayake-pour-lui', 'opera', 'ehsas'],
    'desc': ['floral intense', 'floral intense', 'floral intense', 'natural', 'niche', 'casual', 'niche intense', 'niche', 'niche', 'niche']
})
ACCORD_CLASSES = ['intense', 'floral', 'romantic', 'natural', 'niche']

ACCORD_MAPPING = {
    'floral':   ['floral', 'white floral', 'soft floral', 'rose', 'jasmine'],
    'intense':  ['intense', 'amber', 'oriental', 'warm spicy', 'spicy', 'cinnamon', 'oud'],
    'natural':  ['natural', 'fresh', 'citrus', 'aquatic', 'green', 'fresh spicy', 'aromatic'],
    'niche':    ['niche', 'woody', 'dry wood', 'earthy', 'smoky', 'vetiver', 'sandalwood'],
    'romantic': ['romantic', 'sweet', 'powdery', 'fruity', 'vanilla', 'musky', 'soapy'],
}

# CLIP 텍스트 토큰화
text_inputs = torch.cat([
    clip.tokenize(f"a photo of {perf.replace('-', ' ')} perfume")
    for perf in fra_df['perfume']
]).to(device)

with torch.no_grad():
    perfume_embeddings = clip_model.encode_text(text_inputs)
    perfume_embeddings /= perfume_embeddings.norm(dim=-1, keepdim=True)

# 3. ResNet 모델 가중치 로드
model_eval = MultiTaskResNet18().to(device)
checkpoint_path = os.path.join(PROJECT_ROOT, 'final_model.pth')
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model_dict = model_eval.state_dict()
    pretrained_dict = {k: v for k, v in checkpoint.items() if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(pretrained_dict)
    model_eval.load_state_dict(model_dict)
    print("✅ ResNet 가중치 정밀 동기화 성공.")
model_eval.eval()

if 'style' in mapping_df.columns:
    pure_mapping = mapping_df.drop(columns=['style'])
else:
    pure_mapping = mapping_df
mapping_values = pure_mapping.values  # Shape: (6, 5)

# 데이터를 모으기 위한 리스트 초기화
meta_ids = []
meta_styles = []
meta_r_raw = []
meta_c_raw = []

print("🔄 [Phase 1] 하이브리드 원본 소스 스코어 추출 중...")
with torch.no_grad():
    for imgs, _, ids in tqdm(test_loader, desc="[Extracting Matrix]"):
        imgs = imgs.to(device)

        # ResNet 기반 스타일 및 어코드 연산
        usage_logits, _ = model_eval(imgs)
        resnet_probs = torch.softmax(usage_logits, dim=1).cpu().numpy()
        batch_accord_scores = np.dot(resnet_probs, mapping_values)

        # CLIP 이미지 임베딩 연산
        image_features = clip_model.encode_image(imgs)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        clip_sims = (image_features @ perfume_embeddings.T).cpu().numpy()

        for i in range(len(ids)):
            meta_ids.append(ids[i].item())
            meta_styles.append(STYLE_CLASSES[np.argmax(resnet_probs[i])])

            current_accord_vec = batch_accord_scores[i]
            base_accord_map = {acc: score for acc, score in zip(ACCORD_CLASSES, current_accord_vec)}

            extended_accord_map = {}
            for main_category, sub_keywords in ACCORD_MAPPING.items():
                base_score = base_accord_map.get(main_category, 0.0)
                for kw in sub_keywords:
                    extended_accord_map[kw] = base_score

            r_scores_list_img = []
            for idx, row in fra_df.iterrows():
                words = row['desc'].split()
                r_scores_sub = []
                for w in words:
                    if w in extended_accord_map:
                        r_scores_sub.append(extended_accord_map[w])
                    elif w in base_accord_map:
                        r_scores_sub.append(base_accord_map[w])
                    else:
                        r_scores_sub.append(0.0)
                r_scores_list_img.append(np.mean(r_scores_sub) if len(r_scores_sub) > 0 else 0.0)

            meta_r_raw.append(r_scores_list_img)
            meta_c_raw.append(clip_sims[i])

# 매트릭스 변환
meta_r_raw = np.array(meta_r_raw)
meta_c_raw = np.array(meta_c_raw)

# 1단계 원본 정규화 (전체 기준 평탄화)
r_min, r_max = meta_r_raw.min(), meta_r_raw.max()
c_min, c_max = meta_c_raw.min(), meta_c_raw.max()

meta_r_norm = (meta_r_raw - r_min) / (r_max - r_min + 1e-8)
meta_c_norm = (meta_c_raw - c_min) / (c_max - c_min + 1e-8)

# 2단계: 하이브리드 결합 점수를 전체 리스트로 먼저 확보 (0.95 가둠 깨기 핵심)
total_hybrid_scores = []
for i in range(len(meta_ids)):
    # 최적의 밸런스 가중합 비율 반영 (ResNet 0.45 : CLIP 0.55)
    img_hybrid = (0.45 * meta_r_norm[i]) + (0.55 * meta_c_norm[i])
    total_hybrid_scores.append(img_hybrid)

total_hybrid_scores = np.array(total_hybrid_scores) # Shape: (N, 10)

# ⭐️ [핵심 패치] 이미지 개별 정규화가 아니라 전체 데이터셋의 최소/최대값으로 전역 스케일링 적용!
# 이를 통해 이미지별로 고유한 최종 점수가 보존되어 완벽한 점수 변동성이 확보됩니다.
g_min, g_max = total_hybrid_scores.min(), total_hybrid_scores.max()
total_hybrid_display = 0.73 + (total_hybrid_scores - g_min) / (g_max - g_min + 1e-8) * 0.23

# 3단계: 변동성이 확보된 데이터를 정렬하여 CSV 작성
csv_rows = []
print("⚡ [Phase 2] 전역 변동 스케일 적용 및 최종 랭킹 산출 중...")

for i in range(len(meta_ids)):
    final_scores = total_hybrid_display[i]
    top_indices = np.argsort(final_scores)[::-1]

    csv_rows.append({
        'id': meta_ids[i],
        'highest_predicted_style': meta_styles[i],
        'hybrid_top1_perfume': fra_df.iloc[top_indices[0]]['perfume'],
        'hybrid_top1_score': round(float(final_scores[top_indices[0]]), 4),
        'hybrid_top2_perfume': fra_df.iloc[top_indices[1]]['perfume'] if len(top_indices) > 1 else 'N/A',
        'hybrid_top3_perfume': fra_df.iloc[top_indices[2]]['perfume'] if len(top_indices) > 2 else 'N/A',
        'hybrid_top4_perfume': fra_df.iloc[top_indices[3]]['perfume'] if len(top_indices) > 4 else 'N/A',
        'hybrid_top5_perfume': fra_df.iloc[top_indices[4]]['perfume'] if len(top_indices) > 4 else 'N/A',
    })

# CSV 파일로 최종 저장
output_csv_path = os.path.join(PROJECT_ROOT, 'pred_hybrid_clip_resnet.csv')
pd.DataFrame(csv_rows).to_csv(output_csv_path, index=False, encoding='utf-8-sig')
print(f"🎉 [성공]: {output_csv_path}")

✅ ResNet 가중치 정밀 동기화 성공.
🔄 [Phase 1] 하이브리드 원본 소스 스코어 추출 중...


[Extracting Matrix]: 100%|██████████| 192/192 [00:33<00:00,  5.71it/s]


⚡ [Phase 2] 전역 변동 스케일 적용 및 최종 랭킹 산출 중...
🎉 [성공] /content/drive/MyDrive/FashionPerfumeProject/pred_hybrid_clip_resnet.csv
